In [1]:
import pandas as pd

In [2]:
poi_path = '/home/z5562390/work/dataset2023/cell_POIcat.csv.gz'
cat_path = '/home/z5562390/work/dataset2023/POI_datacategories.csv'

poi_df = pd.read_csv(poi_path)
cat_df = pd.read_csv(cat_path, header=None, names=['category_name'])

print("POI data shape:", poi_df.shape)
print("Category mapping shape:", cat_df.shape)
print()
print(poi_df.head())
print()
print(cat_df.head(10))

POI data shape: (221159, 4)
Category mapping shape: (85, 1)

   x  y  POIcategory  POI_count
0  1  1           48          4
1  1  1           58          1
2  1  1           59          1
3  1  1           69          2
4  1  1           73          1

                category_name
0                        Food
1                    Shopping
2               Entertainment
3         Japanese restaurant
4          Western restaurant
5  Eat all you can restaurant
6          Chinese restaurant
7           Indian restaurant
8            Ramen restaurant
9            Curry restaurant


In [3]:
# 假设 POIcategory 从 1 开始
cat_df['POIcategory'] = cat_df.index + 1

poi_named_df = poi_df.merge(cat_df, on='POIcategory', how='left')

print("Merged POI data:")
print(poi_named_df.head(10))

Merged POI data:
   x  y  POIcategory  POI_count      category_name
0  1  1           48          4    Transit Station
1  1  1           58          1       Kindergarten
2  1  1           59          1        Real Estate
3  1  1           69          2         Hair Salon
4  1  1           73          1   Community Center
5  1  1           74          4             Church
6  1  1           79          2  Building Material
7  1  2           48          1    Transit Station
8  1  2           60          1    Home Appliances
9  1  2           61          2        Post Office


In [4]:
check_ids = [48, 58, 59, 69, 73]

print(cat_df[cat_df['POIcategory'].isin(check_ids)])

       category_name  POIcategory
47   Transit Station           48
57      Kindergarten           58
58       Real Estate           59
68        Hair Salon           69
72  Community Center           73


In [5]:
def build_grid_summary(group, top_k=8):
    group = group.sort_values('POI_count', ascending=False)
    parts = []

    for _, row in group.head(top_k).iterrows():
        cat_name = row['category_name']
        count = int(row['POI_count'])
        parts.append(f"{cat_name}({count})")

    return ", ".join(parts)

In [8]:
grid_summary_df = (
    poi_named_df.groupby(['x', 'y'])
    .apply(build_grid_summary)
    .reset_index(name='poi_summary')
)

print("Grid summary preview:")
print(grid_summary_df.head(10))
print()
print("Number of unique grids:", len(grid_summary_df))

Grid summary preview:
   x   y                                        poi_summary
0  1   1  Transit Station(4), Church(4), Hair Salon(2), ...
1  1   2  Post Office(2), Transit Station(1), Home Appli...
2  1   3  Church(4), Transit Station(2), NPO(2), Heavy I...
3  1   4  Community Center(3), Heavy Industry(2), Transi...
4  1   5  Home Appliances(2), Church(2), Transit Station...
5  1   6  Transit Station(5), Community Center(3), Heavy...
6  1   7  Transit Station(4), Building Material(3), Elde...
7  1   8  Transit Station(3), Western restaurant(2), Chu...
8  1   9  Transit Station(7), Bank(5), Hospital(4), Chur...
9  1  10  NPO(5), Community Center(5), Park(5), Transit ...

Number of unique grids: 20146


/tmp/ipykernel_2937799/2269308737.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_grid_summary)


In [9]:
def classify_grid(summary_text):
    text = summary_text.lower()

    food_keywords = [
        'restaurant', 'cafe', 'bakery', 'bar', 'pizza',
        'sweets', 'diner', 'cuisine', 'tea salon'
    ]
    shopping_keywords = ['shopping', 'retail']
    leisure_keywords = ['entertainment']
    
    food_score = sum(k in text for k in food_keywords)
    shopping_score = sum(k in text for k in shopping_keywords)
    leisure_score = sum(k in text for k in leisure_keywords)

    if food_score >= 2 and shopping_score >= 1:
        return 'mixed_food_commercial'
    elif food_score >= 2:
        return 'food_leisure'
    elif shopping_score >= 1:
        return 'commercial'
    elif leisure_score >= 1:
        return 'entertainment'
    else:
        return 'other'

In [10]:
grid_summary_df['semantic_label'] = grid_summary_df['poi_summary'].apply(classify_grid)

print(grid_summary_df.head(15))

    x   y                                        poi_summary  \
0   1   1  Transit Station(4), Church(4), Hair Salon(2), ...   
1   1   2  Post Office(2), Transit Station(1), Home Appli...   
2   1   3  Church(4), Transit Station(2), NPO(2), Heavy I...   
3   1   4  Community Center(3), Heavy Industry(2), Transi...   
4   1   5  Home Appliances(2), Church(2), Transit Station...   
5   1   6  Transit Station(5), Community Center(3), Heavy...   
6   1   7  Transit Station(4), Building Material(3), Elde...   
7   1   8  Transit Station(3), Western restaurant(2), Chu...   
8   1   9  Transit Station(7), Bank(5), Hospital(4), Chur...   
9   1  10  NPO(5), Community Center(5), Park(5), Transit ...   
10  1  11  Transit Station(4), Japanese restaurant(3), He...   
11  1  12  Heavy Industry(3), Real Estate(1), Church(1), ...   
12  1  13  Hospital(1), Laundry (1), Driving School(1), H...   
13  1  14  Western restaurant(1), Elderly Care Home(1), C...   
14  1  16  Laundry (1), Retail Store(1),

In [11]:
def classify_grid(summary_text):
    text = summary_text.lower()

    food_keywords = [
        'restaurant', 'cafe', 'bakery', 'bar', 'pizza',
        'sweets', 'diner', 'cuisine', 'tea salon'
    ]
    shopping_keywords = ['shopping', 'retail']
    leisure_keywords = ['entertainment']
    
    food_score = sum(k in text for k in food_keywords)
    shopping_score = sum(k in text for k in shopping_keywords)
    leisure_score = sum(k in text for k in leisure_keywords)

    if food_score >= 2 and shopping_score >= 1:
        return 'mixed_food_commercial'
    elif food_score >= 2:
        return 'food_leisure'
    elif shopping_score >= 1:
        return 'commercial'
    elif leisure_score >= 1:
        return 'entertainment'
    else:
        return 'other'

In [12]:
grid_summary_df['semantic_label'] = grid_summary_df['poi_summary'].apply(classify_grid)

print(grid_summary_df.head(15))

    x   y                                        poi_summary  \
0   1   1  Transit Station(4), Church(4), Hair Salon(2), ...   
1   1   2  Post Office(2), Transit Station(1), Home Appli...   
2   1   3  Church(4), Transit Station(2), NPO(2), Heavy I...   
3   1   4  Community Center(3), Heavy Industry(2), Transi...   
4   1   5  Home Appliances(2), Church(2), Transit Station...   
5   1   6  Transit Station(5), Community Center(3), Heavy...   
6   1   7  Transit Station(4), Building Material(3), Elde...   
7   1   8  Transit Station(3), Western restaurant(2), Chu...   
8   1   9  Transit Station(7), Bank(5), Hospital(4), Chur...   
9   1  10  NPO(5), Community Center(5), Park(5), Transit ...   
10  1  11  Transit Station(4), Japanese restaurant(3), He...   
11  1  12  Heavy Industry(3), Real Estate(1), Church(1), ...   
12  1  13  Hospital(1), Laundry (1), Driving School(1), H...   
13  1  14  Western restaurant(1), Elderly Care Home(1), C...   
14  1  16  Laundry (1), Retail Store(1),

In [13]:
for _, row in grid_summary_df.head(5).iterrows():
    print(f"Grid ({row['x']}, {row['y']})")
    print("POI summary:", row['poi_summary'])
    print("Semantic label:", row['semantic_label'])
    print("-" * 60)

Grid (1, 1)
POI summary: Transit Station(4), Church(4), Hair Salon(2), Building Material(2), Kindergarten(1), Real Estate(1), Community Center(1)
Semantic label: other
------------------------------------------------------------
Grid (1, 2)
POI summary: Post Office(2), Transit Station(1), Home Appliances(1), Laundry (1), Gardening(1)
Semantic label: other
------------------------------------------------------------
Grid (1, 3)
POI summary: Church(4), Transit Station(2), NPO(2), Heavy Industry(2), Convenience Store(1), Elderly Care Home(1), Home Appliances(1), Kindergarten(1)
Semantic label: other
------------------------------------------------------------
Grid (1, 4)
POI summary: Community Center(3), Heavy Industry(2), Transit Station(1), Driving School(1), Home Appliances(1), Hair Salon(1), Church(1)
Semantic label: other
------------------------------------------------------------
Grid (1, 5)
POI summary: Home Appliances(2), Church(2), Transit Station(1), Laundry (1), Hair Salon(1),

In [ ]:
demo_x = 1
demo_y = 1

demo_grid_df = poi_named_df[(poi_named_df['x'] == demo_x) & (poi_named_df['y'] == demo_y)]
demo_grid_df = demo_grid_df.sort_values('POI_count', ascending=False)

print(demo_grid_df)